In [ ]:
!pip install -q pyarrow --upgrade
!pip install -q transformers accelerate peft bitsandbytes trl datasets --upgrade
!pip install -q transformers==4.46.0 trl==0.11.4 accelerate peft bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.0 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.3 MB/s eta 0:00:00


In [ ]:
# Core
import torch
import os
import json
import pandas as pd
from tqdm.auto import tqdm

# Transformers & Training
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    AutoModelForSequenceClassification
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)
from trl import SFTTrainer

# Visualization
import matplotlib.pyplot as plt

print("✅ Imports done")

✅ Imports done


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
print(f"✅ GPU Active: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# Paths
WOS_TRAIN = "/content/drive/MyDrive/manual_datasets/KLUE-main/klue_benchmark/wos-v1.1/wos-v1.1_train.json"
WOS_DEV   = "/content/drive/MyDrive/manual_datasets/KLUE-main/klue_benchmark/wos-v1.1/wos-v1.1_dev.json"
ONTOLOGY  = "/content/drive/MyDrive/manual_datasets/KLUE-main/klue_benchmark/wos-v1.1/ontology.json"
SAVE_DIR  = "/content/drive/MyDrive/manual_datasets/dialogue_system/"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ Paths set")

Mounted at /content/drive
✅ GPU Active: True
🖥️ Device: Tesla T4
✅ Paths set


In [ ]:
# Load WoS dataset
with open(WOS_TRAIN, "r", encoding="utf-8") as f:
    wos_train = json.load(f)

with open(WOS_DEV, "r", encoding="utf-8") as f:
    wos_dev = json.load(f)

with open(ONTOLOGY, "r", encoding="utf-8") as f:
    ontology = json.load(f)

print(f"✅ Train dialogues: {len(wos_train)}")
print(f"✅ Dev dialogues:   {len(wos_dev)}")
print(f"✅ Ontology domains: {list(ontology.keys())}")

# Preview one sample to understand the structure
sample = wos_train[0]
print(f"\n--- Sample keys: {list(sample.keys())}")
print(f"--- Domains in sample: {sample.get('domains', [])}")
print(f"--- First turn: {sample['dialogue'][0]}")

✅ Train dialogues: 8000
✅ Dev dialogues:   1000
✅ Ontology domains: ['관광-경치 좋은', '관광-교육적', '관광-도보 가능', '관광-문화 예술', '관광-역사적', '관광-이름', '관광-종류', '관광-주차 가능', '관광-지역', '숙소-가격대', '숙소-도보 가능', '숙소-수영장 유무', '숙소-스파 유무', '숙소-예약 기간', '숙소-예약 명수', '숙소-예약 요일', '숙소-이름', '숙소-인터넷 가능', '숙소-조식 가능', '숙소-종류', '숙소-주차 가능', '숙소-지역', '숙소-헬스장 유무', '숙소-흡연 가능', '식당-가격대', '식당-도보 가능', '식당-야외석 유무', '식당-예약 명수', '식당-예약 시간', '식당-예약 요일', '식당-이름', '식당-인터넷 가능', '식당-종류', '식당-주류 판매', '식당-주차 가능', '식당-지역', '식당-흡연 가능', '지하철-도착지', '지하철-출발 시간', '지하철-출발지', '택시-도착 시간', '택시-도착지', '택시-종류', '택시-출발 시간', '택시-출발지']

--- Sample keys: ['guid', 'domains', 'dialogue']
--- Domains in sample: ['관광', '식당']
--- First turn: {'role': 'user', 'text': '서울 중앙에 있는 박물관을 찾아주세요', 'state': ['관광-종류-박물관', '관광-지역-서울 중앙']}


In [ ]:
def filter_by_domain(data, domain_keyword):
    """Keep only dialogues that contain the target domain."""
    return [d for d in data if any(domain_keyword in dom for dom in d['domains'])]

def format_dialogue(dialogue_data):
    """
    Convert WoS dialogue turns into a single training string
    in Phi-3 Mini instruction format.
    """
    formatted = []

    for convo in dialogue_data:
        turns = convo['dialogue']
        conversation = ""

        for i, turn in enumerate(turns):
            if turn['role'] == 'user':
                conversation += f"<|user|>\n{turn['text']}<|end|>\n"
            else:
                conversation += f"<|assistant|>\n{turn['text']}<|end|>\n"

        formatted.append({"text": conversation.strip()})

    return formatted

# Filter by domain
restaurant_train = filter_by_domain(wos_train, '식당')
restaurant_dev   = filter_by_domain(wos_dev, '식당')
travel_train     = filter_by_domain(wos_train, '관광')
travel_dev       = filter_by_domain(wos_dev, '관광')

print(f"✅ Restaurant train: {len(restaurant_train)} dialogues")
print(f"✅ Restaurant dev:   {len(restaurant_dev)} dialogues")
print(f"✅ Travel train:     {len(travel_train)} dialogues")
print(f"✅ Travel dev:       {len(travel_dev)} dialogues")

# Format for training
restaurant_formatted = format_dialogue(restaurant_train)
travel_formatted     = format_dialogue(travel_train)

print(f"\n--- Example formatted restaurant dialogue:")
print(restaurant_formatted[0]['text'])

✅ Restaurant train: 4688 dialogues
✅ Restaurant dev:   639 dialogues
✅ Travel train:     4318 dialogues
✅ Travel dev:       555 dialogues

--- Example formatted restaurant dialogue:
<|user|>
서울 중앙에 있는 박물관을 찾아주세요<|end|>
<|assistant|>
안녕하세요. 문화역서울 284은 어떠신가요? 평점도 4점으로 방문객들에게 좋은 평가를 받고 있습니다.<|end|>
<|user|>
좋네요 거기 평점은 말해주셨구 전화번호가 어떻게되나요?<|end|>
<|assistant|>
전화번호는 983880764입니다. 더 필요하신 게 있으실까요?<|end|>
<|user|>
네 관광지와 같은 지역의 한식당을 가고싶은데요 야외석이 있어야되요<|end|>
<|assistant|>
생각하고 계신 가격대가 있으신가요?<|end|>
<|user|>
음.. 저렴한 가격대에 있나요?<|end|>
<|assistant|>
죄송하지만 저렴한 가격대에는 없으시네요.<|end|>
<|user|>
그럼 비싼 가격대로 다시 찾아주세요<|end|>
<|assistant|>
외계인의맛집은 어떠신가요? 대표 메뉴는 한정식입니다.<|end|>
<|user|>
좋습니당 토요일 18:00에 1명 예약가능한가요?<|end|>
<|assistant|>
가능합니다. 예약도와드릴까요?<|end|>
<|user|>
넹 거기 주류는 판매하나요?주차는 가능한가요?<|end|>
<|assistant|>
주류는 판매하고 있고 주차도 가능합니다. 더 궁금하신 점 있으신가요?<|end|>
<|user|>
아니용<|end|>
<|assistant|>
감사합니다. 즐거운 여행되세요.<|end|>


In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

# 4-bit quantization config — needed to fit Phi-3 Mini on T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

print(f"✅ Phi-3 Mini loaded")
print(f"📦 Model parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Phi-3 Mini loaded
📦 Model parameters: 2,009,140,224


In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # rank — controls adapter size
    lora_alpha=32,               # scaling factor
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj",
        "up_proj", "down_proj"
    ],
    bias="none"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


In [ ]:
from datasets import Dataset

# Convert to HuggingFace Dataset format
restaurant_dataset = Dataset.from_list(restaurant_formatted)
travel_dataset     = Dataset.from_list(travel_formatted)

# Train/val split for restaurant
restaurant_split = restaurant_dataset.train_test_split(test_size=0.1, seed=42)
travel_split     = travel_dataset.train_test_split(test_size=0.1, seed=42)

print(f"✅ Restaurant — train: {len(restaurant_split['train'])}, val: {len(restaurant_split['test'])}")
print(f"✅ Travel     — train: {len(travel_split['train'])}, val: {len(travel_split['test'])}")

# Preview
print(f"\n--- Example training sample:")
print(restaurant_split['train'][0]['text'])

✅ Restaurant — train: 4219, val: 469
✅ Travel     — train: 3886, val: 432

--- Example training sample:
<|user|>
저 종류는 상관없으니까 서울 동쪽에 적당한 가격대의 식당 좀 찾아주시겠어요?<|end|>
<|assistant|>
안녕하세요, 돈까스 전문점 까스까스까스 확인됩니다. 천호역에서 도보 이용 가능하며 평점도 괜찮은 곳입니다.<|end|>
<|user|>
아 그럼 거기로 예약 좀 해주세요. 화요일 2시 45분에 3명갈거거든요.<|end|>
<|assistant|>
네 예약완료되셨습니다. 관련하여 질문 있으세요?<|end|>
<|user|>
아 식당에서 담배는 필 수 있는지 확인 부탁드리구요, 술은 파는지도 확인해주시겠어요?<|end|>
<|assistant|>
아 해당 식당은 둘 다 안되는 걸로 확인됩니다. <|end|>
<|user|>
아 그래요? 네 알겠습니다. 그럼 서울 시청이요, 거기 영업 시간이랑 경치는 어떤지 좀 확인부탁드리구요, 주차는 되겠죠><|end|>
<|assistant|>
네 먼저 영업 시간은 9시 반에서 17시반까지고요, 경치가 좋은 곳은 아닌 것으로 확인됩니다. 주차도 안되는 걸로 확인되네요. <|end|>
<|user|>
아 잘 알겠습니다. 마지막으로 택시도 예약할건데요, 주점부리에서 신촌역으로 가고, 6시 15분에 출발할거에요. 다른 조건은 상관없으니까 조회 해주시면 되고, 요금이랑 택시 기사분 전화번호도 확인 부탁드려요.<|end|>
<|assistant|>
원하시는 택시 조회됩니다. 신촌역에 6시 반에 도착합니다. 요금은 2만원이며, 기사님 번호는 08493725610입니다. 더 확인하실 부분 있으신가요?<|end|>
<|user|>
아 아뇨 다 됐습니다. 감사합니다.<|end|>
<|assistant|>
네 이용해주셔서 감사합니다.<|end|>


In [ ]:
# Training arguments for Expert 1 (Restaurant)
training_args = TrainingArguments(
    output_dir=os.path.join(SAVE_DIR, "restaurant_expert_checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # effective batch size = 8
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",            # memory efficient optimizer
    lr_scheduler_type="cosine",
    report_to="none",
)

# SFTTrainer for Expert 1
restaurant_trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=restaurant_split['train'],
    eval_dataset=restaurant_split['test'],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

print("✅ Restaurant Expert trainer ready")
print(f"📊 Training samples: {len(restaurant_split['train'])}")
print(f"📊 Validation samples: {len(restaurant_split['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/4219 [00:00<?, ? examples/s]

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

✅ Restaurant Expert trainer ready
📊 Training samples: 4219
📊 Validation samples: 469


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [ ]:
import shutil

# Train Expert 1 — Restaurant
print("🍽️ Training Restaurant Expert...")

# Delete old broken checkpoints
checkpoint_dir = os.path.join(SAVE_DIR, "restaurant_expert_checkpoints")
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print("✅ Old checkpoints deleted")

# Train from scratch
restaurant_trainer.train()

# Save
RESTAURANT_EXPERT_PATH = os.path.join(SAVE_DIR, "restaurant_expert")
restaurant_trainer.model.save_pretrained(RESTAURANT_EXPERT_PATH)
tokenizer.save_pretrained(RESTAURANT_EXPERT_PATH)

print(f"✅ Restaurant Expert saved to: {RESTAURANT_EXPERT_PATH}")

🍽️ Training Restaurant Expert...
✅ Old checkpoints deleted


Step,Training Loss
25,2.045200
50,2.034100
75,2.079600
100,2.113800
125,1.976000
150,2.024800
175,2.023300
200,1.975300
225,1.913400
250,1.980000


✅ Restaurant Expert saved to: /content/drive/MyDrive/manual_datasets/dialogue_system/restaurant_expert


In [ ]:
# ⚠️ DO NOT RUN — Travel Expert already trained and saved to Drive
# Kept for documentation purposes only
# TRAVEL_EXPERT_PATH = /content/drive/MyDrive/manual_datasets/dialogue_system/travel_expert

# Train Expert 2 — Travel (full retrain, 3 epochs)
print("🗺️ Training Travel Expert...")

# Clear old travel checkpoints first
import shutil
travel_checkpoint_dir = os.path.join(SAVE_DIR, "travel_expert_checkpoints")
if os.path.exists(travel_checkpoint_dir):
    shutil.rmtree(travel_checkpoint_dir)
    print("✅ Old travel checkpoints cleared")

# Fresh LoRA config for travel expert
travel_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj",
        "up_proj", "down_proj"
    ],
    bias="none"
)

# Load fresh base model for travel expert
travel_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
travel_base_model.config.use_cache = False
travel_base_model = get_peft_model(travel_base_model, travel_lora_config)
travel_base_model.print_trainable_parameters()

training_args_travel = TrainingArguments(
    output_dir=travel_checkpoint_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
)

travel_trainer = SFTTrainer(
    model=travel_base_model,
    args=training_args_travel,
    train_dataset=travel_split['train'],
    eval_dataset=travel_split['test'],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

travel_trainer.train()

# Save
TRAVEL_EXPERT_PATH = os.path.join(SAVE_DIR, "travel_expert")
travel_trainer.model.save_pretrained(TRAVEL_EXPERT_PATH)
tokenizer.save_pretrained(TRAVEL_EXPERT_PATH)

print(f"✅ Travel Expert saved to: {TRAVEL_EXPERT_PATH}")

🗺️ Training Travel Expert...
✅ Old travel checkpoints cleared


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/3886 [00:00<?, ? examples/s]

Map:   0%|          | 0/432 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Step,Training Loss
25,5.890800
50,4.434400
75,3.662500
100,3.358200
125,3.158400
150,2.998300
175,2.894600
200,2.863500
225,2.787100
250,2.606200


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# Build intent classification dataset from WoS
def build_intent_dataset(data):
    samples = []
    for convo in data:
        domains = convo['domains']
        # Get first user turn as the input
        first_turn = convo['dialogue'][0]['text']

        # Label: 0 = restaurant, 1 = travel
        if '식당' in domains and '관광' not in domains:
            label = 0  # restaurant only
        elif '관광' in domains and '식당' not in domains:
            label = 1  # travel only
        else:
            continue   # skip mixed domain dialogues

        samples.append({"text": first_turn, "label": label})
    return samples

train_intent = build_intent_dataset(wos_train)
dev_intent   = build_intent_dataset(wos_dev)

print(f"✅ Intent train samples: {len(train_intent)}")
print(f"✅ Intent dev samples:   {len(dev_intent)}")
print(f"📊 Restaurant (0): {sum(1 for s in train_intent if s['label'] == 0)}")
print(f"📊 Travel (1):     {sum(1 for s in train_intent if s['label'] == 1)}")

# Preview
print(f"--- Restaurant example: {next(s for s in train_intent if s['label'] == 0)}")
print(f"--- Travel example:     {next(s for s in train_intent if s['label'] == 1)}")

✅ Intent train samples: 4402
✅ Intent dev samples:   566
📊 Restaurant (0): 2386
📊 Travel (1):     2016
--- Restaurant example: {'text': '안녕하세요. 서울 북쪽에 주차가 가능한 중식당을 찾고 있습니다.', 'label': 0}
--- Travel example:     {'text': '쇼핑을 하려는데 서울 서쪽에 있을까요?', 'label': 1}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

ROUTER_MODEL_ID = "klue/roberta-small"

# Load tokenizer and model
router_tokenizer = AutoTokenizer.from_pretrained(ROUTER_MODEL_ID)
router_model = AutoModelForSequenceClassification.from_pretrained(
    ROUTER_MODEL_ID,
    num_labels=2
).to("cuda")

# Dataset class
class IntentDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=64):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        encoding = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

# Create datasets and dataloaders
train_dataset = IntentDataset(train_intent, router_tokenizer)
dev_dataset   = IntentDataset(dev_intent, router_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=32)

print(f"✅ Router model loaded: {ROUTER_MODEL_ID}")
print(f"📊 Train batches: {len(train_loader)}")
print(f"📊 Dev batches:   {len(dev_loader)}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/971 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/273M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Router model loaded: klue/roberta-small
📊 Train batches: 138
📊 Dev batches:   18


In [ ]:
from torch.optim import AdamW

# Training loop for intent router
optimizer = AdamW(router_model.parameters(), lr=2e-5)
num_epochs = 3

print("🧭 Training Intent Router...")

for epoch in range(num_epochs):
    # Training
    router_model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        input_ids      = batch['input_ids'].to("cuda")
        attention_mask = batch['attention_mask'].to("cuda")
        labels         = batch['label'].to("cuda")

        optimizer.zero_grad()
        outputs = router_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs[0]
        logits  = outputs[1]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total * 100
    avg_loss  = total_loss / len(train_loader)

    # Validation — pass labels to get 2 outputs
    router_model.eval()
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for batch in dev_loader:
            input_ids      = batch['input_ids'].to("cuda")
            attention_mask = batch['attention_mask'].to("cuda")
            labels         = batch['label'].to("cuda")

            outputs  = router_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            preds    = torch.argmax(outputs[1], dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    val_acc = val_correct / val_total * 100
    print(f"Epoch {epoch+1} — Loss: {avg_loss:.4f} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

# Save router to Drive
ROUTER_PATH = os.path.join(SAVE_DIR, "intent_router")
router_model.save_pretrained(ROUTER_PATH)
router_tokenizer.save_pretrained(ROUTER_PATH)
print(f"\n✅ Intent Router saved to: {ROUTER_PATH}")

🧭 Training Intent Router...


Epoch 1/3:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 1 — Loss: 0.1738 | Train Acc: 90.8% | Val Acc: 90.8%


Epoch 2/3:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 2 — Loss: 0.1339 | Train Acc: 92.9% | Val Acc: 91.3%


Epoch 3/3:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 3 — Loss: 0.1145 | Train Acc: 94.0% | Val Acc: 93.1%

✅ Intent Router saved to: /content/drive/MyDrive/manual_datasets/dialogue_system/intent_router


In [ ]:
def route_intent(text):
    """Route input to the correct expert using the trained classifier."""
    router_model.eval()
    encoding = router_tokenizer(
        text,
        max_length=64,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to("cuda")
    attention_mask = encoding['attention_mask'].to("cuda")

    with torch.no_grad():
        outputs = router_model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids.new_zeros(1))
        logits  = outputs[1]

    probs  = torch.softmax(logits, dim=-1)
    pred   = torch.argmax(probs, dim=-1).item()
    score  = probs[0][pred].item()

    intent = "restaurant" if pred == 0 else "travel"
    return intent, score

# Test
test_inputs = [
    "예약하고 싶어요",
    "박물관을 찾아주세요",
    "메뉴가 어떻게 되나요?",
    "서울 관광지 추천해주세요",
    "야외석이 있는 식당 찾아주세요",
    "경복궁 입장료가 얼마예요?",
]

print("🧭 Intent Router Test:")
for text in test_inputs:
    intent, score = route_intent(text)
    print(f"  '{text}' → {intent} ({score*100:.1f}%)")

🧭 Intent Router Test:
  '예약하고 싶어요' → travel (93.8%)
  '박물관을 찾아주세요' → travel (99.9%)
  '메뉴가 어떻게 되나요?' → restaurant (99.3%)
  '서울 관광지 추천해주세요' → travel (99.9%)
  '야외석이 있는 식당 찾아주세요' → restaurant (99.9%)
  '경복궁 입장료가 얼마예요?' → travel (99.9%)


In [ ]:
from peft import PeftModel

def load_expert(expert_path):
    """Load a LoRA expert on a fresh base model each time."""
    fresh_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    fresh_model.config.use_cache = False
    return PeftModel.from_pretrained(fresh_model, expert_path)

def generate_response(model, tokenizer, user_input, max_new_tokens=200):
    """Generate a response from the active expert."""
    prompt = f"<|user|>\n{user_input}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def moe_respond(user_input, tokenizer):
    """
    Full MoE pipeline:
    1. Route intent
    2. Load correct expert
    3. Generate response
    """
    intent, confidence = route_intent(user_input)
    print(f"🧭 Routed to: {intent} expert ({confidence*100:.1f}% confidence)")

    expert_path = RESTAURANT_EXPERT_PATH if intent == "restaurant" else TRAVEL_EXPERT_PATH
    expert_model = load_expert(expert_path)

    response = generate_response(expert_model, tokenizer, user_input)
    return response, intent

print("✅ MoE wrapper updated")

✅ MoE wrapper updated


In [ ]:
# End-to-end MoE test
test_inputs = [
    "야외석이 있는 식당 찾아주세요",
    "서울 북쪽에 박물관이 있나요?",
    "메뉴가 어떻게 되나요?",
    "경복궁 근처 관광지 추천해주세요",
]

print("\n🤖 MoE Dialogue System — End to End Test\n")
for user_input in test_inputs:
    print(f"👤 User: {user_input}")
    response, intent = moe_respond(user_input, tokenizer)
    print(f"🤖 [{intent.upper()} EXPERT]: {response}")
    print("-" * 60)


🤖 MoE Dialogue System — End to End Test

👤 User: 야외석이 있는 식당 찾아주세요
🧭 Routed to: restaurant expert (99.9% confidence)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🤖 [RESTAURANT EXPERT]: 안녕하세요. 식당의 가격대와 종류, 지역은 어떻게 도와드릴까요?
------------------------------------------------------------
👤 User: 서울 북쪽에 박물관이 있나요?
🧭 Routed to: travel expert (99.9% confidence)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🤖 [TRAVEL EXPERT]: 이런 질문을 풀기 위해서는 서울 북쪽에 있는 박물관을 설명하거나 지도로 찾아보는 것이 좋습니다. 이 분야에서 있는 주요한 박물관은 아래와 같습니다:

1. 서울북쪽 장악대: 축구 박물관
2. 서울북쪽 석당대: 신문사 박�����
------------------------------------------------------------
👤 User: 메뉴가 어떻게 되나요?
🧭 Routed to: restaurant expert (99.3% confidence)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🤖 [RESTAURANT EXPERT]: 안녕하세요? 네, 반갑습니다. 먼저 네, 건대입구역에서 가까운 식당 예약 도와드리면 될까요?
------------------------------------------------------------
👤 User: 경복궁 근처 관광지 추천해주세요
🧭 Routed to: travel expert (99.9% confidence)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🤖 [TRAVEL EXPERT]: 경복궁 근처 관관지 추천은 다음과 같습니다:

1. 경복궁 이전 근처 관괴지 출루 - 경복궁 이전 근처 관괴지 중 하나는 관괴후 위 경융궁에서 알 수 émaile 길이 길이가 길다는 것을 망침과 근처 관괴지에서도 있습니
------------------------------------------------------------


In [ ]:
!pip install jamo